In [ ]:
import json
import os
from pathlib import Path

# Find all JSON files in the data/ directory
data_dir = Path("data/")
json_files = list(data_dir.glob("*.json"))

print(f"Found {len(json_files)} JSON files in data/")
print("=" * 50)

total_items = 0

# Load each JSON file and show the number of items
for json_file in sorted(json_files):
    try:
        with open(json_file, 'r') as f:
            data = json.load(f)
        
        # Handle different data structures
        if isinstance(data, list):
            num_items = len(data)
        elif isinstance(data, dict):
            num_items = len(data)
        else:
            num_items = 1
            
        print(f"{json_file.name}: {num_items} items")
        total_items += num_items
        
    except Exception as e:
        print(f"{json_file.name}: Error loading file - {e}")

print("=" * 50)
print(f"Total items across all JSON files: {total_items}")



In [2]:
from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor
from qwen_vl_utils import process_vision_info

# You can set the maximum tokens for a video through the environment variable VIDEO_MAX_PIXELS
# based on the maximum tokens that the model can accept. 
# export VIDEO_MAX_PIXELS = 32000 * 28 * 28 * 0.9


# You can directly insert a local file path, a URL, or a base64-encoded image into the position where you want in the text.
messages = [
    # # Image
    # ## Local file path
    # [{"role": "user", "content": [{"type": "image", "image": "file:///path/to/your/image.jpg"}, {"type": "text", "text": "Describe this image."}]}],
    # ## Image URL
    # [{"role": "user", "content": [{"type": "image", "image": "http://path/to/your/image.jpg"}, {"type": "text", "text": "Describe this image."}]}],
    # ## Base64 encoded image
    # [{"role": "user", "content": [{"type": "image", "image": "data:image;base64,/9j/..."}, {"type": "text", "text": "Describe this image."}]}],
    # ## PIL.Image.Image
    # [{"role": "user", "content": [{"type": "image", "image": pil_image}, {"type": "text", "text": "Describe this image."}]}],
    # ## Model dynamically adjusts image size, specify dimensions if required.
    # [{"role": "user", "content": [{"type": "image", "image": "file:///path/to/your/image.jpg", "resized_height": 280, "resized_width": 420}, {"type": "text", "text": "Describe this image."}]}],
    # # Video
    # ## Local video path
    # [{"role": "user", "content": [{"type": "video", "video": "file:///path/to/video1.mp4"}, {"type": "text", "text": "Describe this video."}]}],
    # ## Local video frames
    # [{"role": "user", "content": [{"type": "video", "video": ["file:///path/to/extracted_frame1.jpg", "file:///path/to/extracted_frame2.jpg", "file:///path/to/extracted_frame3.jpg"],}, {"type": "text", "text": "Describe this video."},],}],
    # ## Model dynamically adjusts video nframes, video height and width. specify args if required.
    [{"role": "user", "content": [{"type": "video", "video": "/home/ec2-user/SageMaker/efs/Projects/vlm-rl-training/data/video/kids_videos/a_baby_crawling_on_the_floor.mp4", "fps": 2.0, "resized_height": 280, "resized_width": 280}, {"type": "text", "text": "Describe this video."}]}],
    [{"role": "user", "content": [{"type": "video", "video": "/home/ec2-user/SageMaker/efs/Projects/vlm-rl-training/data/video/kids_videos/a_baby_eating_from_a_white_bowl_at_a_table.mp4", "fps": 2.0, "resized_height": 280, "resized_width": 280}, {"type": "text", "text": "Describe this video."}]}],
]

model_path = "/home/ec2-user/SageMaker/efs/Models/Qwen2.5-VL-7B-Instruct"

processor = AutoProcessor.from_pretrained(model_path)
print(f"processor: {processor}")
model = Qwen2_5_VLForConditionalGeneration.from_pretrained(model_path, torch_dtype="auto", device_map="auto")

The image processor of type `Qwen2VLImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. Note that this behavior will be extended to all models in a future release.


You have video processor config saved in `preprocessor.json` file which is deprecated. Video processor configs should be saved in their own `video_preprocessor.json` file. You can rename the file or load and save the processor back which renames it automatically. Loading from `preprocessor.json` will be removed in v5.0.


processor: Qwen2_5_VLProcessor:
- image_processor: Qwen2VLImageProcessorFast {
  "crop_size": null,
  "data_format": "channels_first",
  "default_to_square": true,
  "device": null,
  "disable_grouping": null,
  "do_center_crop": null,
  "do_convert_rgb": true,
  "do_normalize": true,
  "do_rescale": true,
  "do_resize": true,
  "image_mean": [
    0.48145466,
    0.4578275,
    0.40821073
  ],
  "image_processor_type": "Qwen2VLImageProcessorFast",
  "image_std": [
    0.26862954,
    0.26130258,
    0.27577711
  ],
  "input_data_format": null,
  "max_pixels": 12845056,
  "merge_size": 2,
  "min_pixels": 3136,
  "patch_size": 14,
  "processor_class": "Qwen2_5_VLProcessor",
  "resample": 3,
  "rescale_factor": 0.00392156862745098,
  "return_tensors": null,
  "size": {
    "longest_edge": 12845056,
    "shortest_edge": 3136
  },
  "temporal_patch_size": 2
}

- tokenizer: Qwen2TokenizerFast(name_or_path='/home/ec2-user/SageMaker/efs/Models/Qwen2.5-VL-7B-Instruct', vocab_size=151643, model

Loading checkpoint shards: 100%|██████████| 5/5 [17:47<00:00, 213.57s/it]


In [3]:
print(f"messages: {messages}")
text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
print(f"text: {text}")
images, videos, video_kwargs = process_vision_info(messages, return_video_kwargs=True)
print(f"images: {images}\n\nvideos: {type(videos)}, {len(videos)}, {videos[0].shape}, {videos}\n\nvideo_args: {video_kwargs}")

inputs = processor(text=text, images=images, videos=videos, padding=True, return_tensors="pt", **video_kwargs)
inputs = inputs.to("cuda:0")
print(f"inputs: {inputs}")

generated_ids = model.generate(**inputs)
print(f"generated_ids: {generated_ids}")

generated_ids_trimmed = [
    out_ids[len(in_ids) :] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
]
output_text = processor.batch_decode(
    generated_ids_trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False
)
print(f"output_text: {output_text}")

messages: [[{'role': 'user', 'content': [{'type': 'video', 'video': '/home/ec2-user/SageMaker/efs/Projects/vlm-rl-training/data/video/kids_videos/a_baby_crawling_on_the_floor.mp4', 'fps': 2.0, 'resized_height': 280, 'resized_width': 280}, {'type': 'text', 'text': 'Describe this video.'}]}], [{'role': 'user', 'content': [{'type': 'video', 'video': '/home/ec2-user/SageMaker/efs/Projects/vlm-rl-training/data/video/kids_videos/a_baby_eating_from_a_white_bowl_at_a_table.mp4', 'fps': 2.0, 'resized_height': 280, 'resized_width': 280}, {'type': 'text', 'text': 'Describe this video.'}]}]]
text: ['<|im_start|>system\nYou are a helpful assistant.<|im_end|>\n<|im_start|>user\n<|vision_start|><|video_pad|><|vision_end|>Describe this video.<|im_end|>\n<|im_start|>assistant\n', '<|im_start|>system\nYou are a helpful assistant.<|im_end|>\n<|im_start|>user\n<|vision_start|><|video_pad|><|vision_end|>Describe this video.<|im_end|>\n<|im_start|>assistant\n']


qwen-vl-utils using decord to read video.


images: None

videos: <class 'list'>, 2, torch.Size([28, 3, 280, 280]), [tensor([[[[79., 80., 81.,  ..., 36., 36., 36.],
          [83., 84., 85.,  ..., 32., 31., 30.],
          [90., 90., 90.,  ..., 30., 30., 30.],
          ...,
          [61., 61., 61.,  ..., 15., 15., 15.],
          [61., 61., 61.,  ..., 15., 15., 15.],
          [61., 61., 61.,  ..., 15., 15., 15.]],

         [[33., 34., 35.,  ..., 24., 24., 24.],
          [37., 38., 39.,  ..., 20., 19., 18.],
          [44., 44., 44.,  ..., 18., 18., 18.],
          ...,
          [41., 41., 41.,  ..., 10., 10., 10.],
          [41., 41., 41.,  ..., 10., 10., 10.],
          [41., 41., 41.,  ..., 10., 10., 10.]],

         [[17., 18., 19.,  ..., 24., 24., 24.],
          [21., 22., 23.,  ..., 20., 19., 18.],
          [28., 28., 28.,  ..., 18., 18., 18.],
          ...,
          [34., 34., 34.,  ..., 12., 12., 12.],
          [34., 34., 34.,  ..., 12., 12., 12.],
          [34., 34., 34.,  ..., 12., 12., 12.]]],


        [[

A decoder-only architecture is being used, but right-padding was detected! For correct generation results, please set `padding_side='left'` when initializing the tokenizer.


inputs: {'input_ids': tensor([[151644,   8948,    198,  ..., 151644,  77091,    198],
        [151644,   8948,    198,  ..., 151643, 151643, 151643]],
       device='cuda:0'), 'attention_mask': tensor([[1, 1, 1,  ..., 1, 1, 1],
        [1, 1, 1,  ..., 0, 0, 0]], device='cuda:0'), 'pixel_values_videos': tensor([[-0.6390, -0.6244, -0.6098,  ..., -1.2669, -1.2811, -1.2811],
        [-0.4930, -0.4930, -0.4930,  ..., -1.2954, -1.2954, -1.2954],
        [-0.8288, -0.8288, -0.8288,  ..., -1.4802, -1.4802, -1.4802],
        ...,
        [ 1.6092,  1.6092,  1.5946,  ...,  1.7904,  1.7762,  1.7762],
        [ 1.6238,  1.6092,  1.6092,  ...,  0.6955,  0.7097,  0.7239],
        [ 1.5946,  1.5946,  1.5800,  ...,  0.7381,  0.7381,  0.8661]],
       device='cuda:0'), 'video_grid_thw': tensor([[14, 20, 20],
        [ 9, 20, 20]], device='cuda:0'), 'second_per_grid_ts': tensor([1.0158, 1.0481], device='cuda:0')}
generated_ids: tensor([[151644,   8948,    198,  ...,  41199,     11,   6319],
        [151

In [4]:
inputs['input_ids'].shape

torch.Size([2, 1425])

In [17]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(model_path)
tokens = tokenizer.tokenize(text[0])
print(len(tokens), tokens)

26 ['<|im_start|>', 'system', 'Ċ', 'You', 'Ġare', 'Ġa', 'Ġhelpful', 'Ġassistant', '.', '<|im_end|>', 'Ċ', '<|im_start|>', 'user', 'Ċ', '<|vision_start|>', '<|video_pad|>', '<|vision_end|>', 'Describe', 'Ġthis', 'Ġvideo', '.', '<|im_end|>', 'Ċ', '<|im_start|>', 'assistant', 'Ċ']


In [ ]:
inputs['input_ids'].shape

In [ ]:
inputs['pixel_values_videos'].shape

In [ ]:
5600*1176

In [ ]:
28*3*280*280


In [21]:
input_text = processor.batch_decode(
    inputs.input_ids, skip_special_tokens=False, clean_up_tokenization_spaces=False
)

print(f"input_text: {input_text}")

input_text: ['<|im_start|>system\nYou are a helpful assistant.<|im_end|>\n<|im_start|>user\n<|vision_start|><|video_pad|><|video_pad|><|video_pad|><|video_pad|><|video_pad|><|video_pad|><|video_pad|><|video_pad|><|video_pad|><|video_pad|><|video_pad|><|video_pad|><|video_pad|><|video_pad|><|video_pad|><|video_pad|><|video_pad|><|video_pad|><|video_pad|><|video_pad|><|video_pad|><|video_pad|><|video_pad|><|video_pad|><|video_pad|><|video_pad|><|video_pad|><|video_pad|><|video_pad|><|video_pad|><|video_pad|><|video_pad|><|video_pad|><|video_pad|><|video_pad|><|video_pad|><|video_pad|><|video_pad|><|video_pad|><|video_pad|><|video_pad|><|video_pad|><|video_pad|><|video_pad|><|video_pad|><|video_pad|><|video_pad|><|video_pad|><|video_pad|><|video_pad|><|video_pad|><|video_pad|><|video_pad|><|video_pad|><|video_pad|><|video_pad|><|video_pad|><|video_pad|><|video_pad|><|video_pad|><|video_pad|><|video_pad|><|video_pad|><|video_pad|><|video_pad|><|video_pad|><|video_pad|><|video_pad|><|video_

In [ ]:
%export VLLM_WORKER_MULTIPROC_METHOD=spawn

UsageError: Line magic function `%` not found.


In [9]:
!echo $VLLM_WORKER_MULTIPROC_METHOD

In [1]:
from transformers import AutoProcessor
from vllm import LLM, SamplingParams
from qwen_vl_utils import process_vision_info

# import multiprocessing as mp
# mp.set_start_method('spawn', force=True)
import os

os.environ["VLLM_WORKER_MULTIPROC_METHOD"] = "spawn"


MODEL_PATH = "/home/ec2-user/SageMaker/efs/Models/Qwen2.5-VL-7B-Instruct"

llm = LLM(
    model=MODEL_PATH,
    limit_mm_per_prompt={"image": 10, "video": 10},
)


/home/ec2-user/SageMaker/efs/conda_envs/trl/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


INFO 08-22 05:54:08 [__init__.py:241] Automatically detected platform cuda.
INFO 08-22 05:54:10 [utils.py:326] non-default args: {'model': '/home/ec2-user/SageMaker/efs/Models/Qwen2.5-VL-7B-Instruct', 'disable_log_stats': True, 'limit_mm_per_prompt': {'image': 10, 'video': 10}}
INFO 08-22 05:54:23 [__init__.py:711] Resolved architecture: Qwen2_5_VLForConditionalGeneration
INFO 08-22 05:54:23 [__init__.py:1750] Using max model len 128000


2025-08-22 05:54:23,967	INFO util.py:154 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.


INFO 08-22 05:54:23 [scheduler.py:222] Chunked prefill is enabled with max_num_batched_tokens=16384.
INFO 08-22 05:54:36 [__init__.py:241] Automatically detected platform cuda.
(EngineCore_0 pid=421777) INFO 08-22 05:54:38 [core.py:636] Waiting for init message from front-end.
(EngineCore_0 pid=421777) INFO 08-22 05:54:38 [core.py:74] Initializing a V1 LLM engine (v0.10.1.1) with config: model='/home/ec2-user/SageMaker/efs/Models/Qwen2.5-VL-7B-Instruct', speculative_config=None, tokenizer='/home/ec2-user/SageMaker/efs/Models/Qwen2.5-VL-7B-Instruct', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, override_neuron_config={}, tokenizer_revision=None, trust_remote_code=False, dtype=torch.bfloat16, max_seq_len=128000, download_dir=None, load_format=auto, tensor_parallel_size=1, pipeline_parallel_size=1, disable_custom_all_reduce=False, quantization=None, enforce_eager=False, kv_cache_dtype=auto, device_config=cuda, decoding_config=DecodingConfig(backend='auto', disable_fallba

(EngineCore_0 pid=421777) You have video processor config saved in `preprocessor.json` file which is deprecated. Video processor configs should be saved in their own `video_preprocessor.json` file. You can rename the file or load and save the processor back which renames it automatically. Loading from `preprocessor.json` will be removed in v5.0.


(EngineCore_0 pid=421777) INFO 08-22 05:54:43 [gpu_model_runner.py:1953] Starting to load model /home/ec2-user/SageMaker/efs/Models/Qwen2.5-VL-7B-Instruct...
(EngineCore_0 pid=421777) INFO 08-22 05:54:43 [gpu_model_runner.py:1985] Loading model from scratch...
(EngineCore_0 pid=421777) INFO 08-22 05:54:43 [cuda.py:328] Using Flash Attention backend on V1 engine.


Loading safetensors checkpoint shards:   0% Completed | 0/5 [00:00<?, ?it/s]
Loading safetensors checkpoint shards:  20% Completed | 1/5 [00:00<00:02,  1.36it/s]
Loading safetensors checkpoint shards:  40% Completed | 2/5 [00:00<00:01,  2.18it/s]
Loading safetensors checkpoint shards:  60% Completed | 3/5 [00:01<00:01,  1.64it/s]
Loading safetensors checkpoint shards:  80% Completed | 4/5 [00:02<00:00,  1.47it/s]
Loading safetensors checkpoint shards: 100% Completed | 5/5 [00:03<00:00,  1.40it/s]
Loading safetensors checkpoint shards: 100% Completed | 5/5 [00:03<00:00,  1.49it/s]
(EngineCore_0 pid=421777) 


(EngineCore_0 pid=421777) INFO 08-22 05:54:47 [default_loader.py:262] Loading weights took 3.40 seconds
(EngineCore_0 pid=421777) INFO 08-22 05:54:47 [gpu_model_runner.py:2007] Model loading took 15.6269 GiB and 3.750132 seconds
(EngineCore_0 pid=421777) INFO 08-22 05:54:47 [gpu_model_runner.py:2591] Encoder cache will be initialized with a budget of 98304 tokens, and profiled with 1 video items of the maximum feature size.
(EngineCore_0 pid=421777) INFO 08-22 05:54:57 [backends.py:548] Using cache directory: /home/ubuntu/.cache/vllm/torch_compile_cache/c7dd04bd4a/rank_0_0/backbone for vLLM's torch.compile
(EngineCore_0 pid=421777) INFO 08-22 05:54:57 [backends.py:559] Dynamo bytecode transform time: 6.61 s
(EngineCore_0 pid=421777) INFO 08-22 05:55:00 [backends.py:194] Cache the graph for dynamic shape for later use
(EngineCore_0 pid=421777) INFO 08-22 05:55:14 [backends.py:215] Compiling a graph for dynamic shape takes 17.10 s
(EngineCore_0 pid=421777) INFO 08-22 05:55:19 [monitor.py

Capturing CUDA graphs (mixed prefill-decode, PIECEWISE): 100%|██████████| 67/67 [00:02<00:00, 25.61it/s]


(EngineCore_0 pid=421777) INFO 08-22 05:55:23 [gpu_model_runner.py:2708] Graph capturing finished in 3 secs, took 0.69 GiB
(EngineCore_0 pid=421777) INFO 08-22 05:55:23 [core.py:214] init engine (profile, create kv cache, warmup model) took 35.45 seconds
INFO 08-22 05:55:24 [llm.py:298] Supported_tasks: ['generate']


In [1]:

sampling_params = SamplingParams(
    temperature=0.1,
    top_p=0.001,
    repetition_penalty=1.05,
    max_tokens=256,
    stop_token_ids=[],
)

image_messages = [
    {"role": "system", "content": "You are a helpful assistant."},
    {
        "role": "user",
        "content": [
            {
                "type": "image",
                "image": "https://modelscope.oss-cn-beijing.aliyuncs.com/resource/qwen.png",
                "min_pixels": 224 * 224,
                "max_pixels": 1280 * 28 * 28,
            },
            {"type": "text", "text": "What is the text in the illustrate?"},
        ],
    },
]


# For video input, you can pass following values instead:
# "type": "video",
# "video": "<video URL>",
video_messages = [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": [
            {"type": "text", "text": "请用表格总结一下视频中的商品特点"},
            {
                "type": "video", 
                "video": "https://duguang-labelling.oss-cn-shanghai.aliyuncs.com/qiansun/video_ocr/videos/50221078283.mp4",
                "total_pixels": 20480 * 28 * 28, "min_pixels": 16 * 28 * 28
            }
        ]
    },
]

# Here we use video messages as a demonstration
messages = video_messages
# messages = image_messages

processor = AutoProcessor.from_pretrained(MODEL_PATH)
prompt = processor.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
)
image_inputs, video_inputs, video_kwargs = process_vision_info(messages, return_video_kwargs=True)

mm_data = {}
if image_inputs is not None:
    mm_data["image"] = image_inputs
if video_inputs is not None:
    mm_data["video"] = video_inputs

llm_inputs = {
    "prompt": prompt,
    "multi_modal_data": mm_data,

    # FPS will be returned in video_kwargs
    "mm_processor_kwargs": video_kwargs,
}

print(f"llm_inputs: {llm_inputs}")

NameError: name 'SamplingParams' is not defined

In [18]:
print("image_token", getattr(processor, "image_token", None))
print("image_token_id", getattr(processor, "image_token_id", None))
print("video_token", getattr(processor, "video_token", None))
print("video_token_id", getattr(processor, "video_token_id", None))
# print("vision_start_token_id", getattr(llm.config, "vision_start_token_id", None))
# print("vision_end_token_id", getattr(llm.config, "vision_end_token_id", None))

image_token <|image_pad|>
image_token_id 151655
video_token <|video_pad|>
video_token_id 151656


In [7]:
mm_data['video'][0].shape

torch.Size([58, 3, 532, 980])

In [10]:
# mm_data["image"]
video_kwargs

{'fps': [1.9568151147098516]}

In [3]:
outputs = llm.generate([llm_inputs], sampling_params=sampling_params)
generated_text = outputs[0].outputs[0].text

print(generated_text)


Processed prompts: 100%|██████████| 1/1 [00:03<00:00,  3.06s/it, est. speed input: 6304.29 toks/s, output: 63.65 toks/s]

以下是根据视频内容总结的商品特点表格：

| 特点 | 描述 |
|------|------|
| 适用范围广 | 可用于龙眼、切片西瓜、圣女果、樱桃等多种水果包装。 |
| 捏扣设计 | 人性化设计，易扣不繁琐。 |
| 捏扣紧锁 | 上下盖紧锁，摇晃不脱落。 |
| 专业铝膜 | 采用PET材料制作，做工精细。 |
| 防压抗摔 | 耐压耐磨，耐低温，可冷藏。 |
| 美观实用 | 纹理清晰质感佳，形状好，光泽度好。 |
| 高透加厚 | 盒内产品一目了然，无色无味。 |
| 全面展示 | 全面展示产品细节。 |

希望这个表格能帮助你更好地理解视频中商品的特点！
